# 🧪 RAG Agent Evaluation with LLM-as-Judge

## What this notebook does
This notebook evaluates your **Mechanical Engineering RAG Agent** using **4 types of LLM-as-judge metrics**, all visible in your **LangSmith dashboard**.

---

## 📐 The 4 Evaluation Metrics (from lecture)

| # | Metric | What it measures | Needs reference answer? |
|---|--------|-----------------|------------------------|
| 1 | **Answer Accuracy** | Is the answer correct vs. a reference answer? | ✅ Yes |
| 2 | **Answer Hallucination** | Is the answer grounded in retrieved docs? | ❌ No |
| 3 | **Document Relevance** | Are the retrieved docs relevant to the question? | ❌ No |
| 4 | **Answer Helpfulness** | Does the answer address the user's question? | ❌ No |

---

## 🔧 What YOU need to change before running
There are **2 things** you must customize:

1. **Section 3**: Add your own eval questions + reference answers (5-10 QA pairs)
2. **Section 2**: Your `.env` file must have the same keys as your RAG agent notebook

Everything else is plug-and-play ✅

## 1. Install & Import Dependencies

In [ ]:
# Run this cell ONCE to install required packages
# !pip install langchain langchain-openai langchain-pinecone langgraph langsmith python-dotenv pydantic

In [1]:
import os
import json
from typing import List, Dict, Any
from datetime import datetime

from dotenv import load_dotenv

# LangChain Core
from langchain.schema import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.tools import StructuredTool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain import hub

# LangSmith Evaluation
from langsmith import Client
from langsmith.evaluation import evaluate
from langsmith import traceable

# Pydantic
from pydantic import BaseModel, Field

print("✅ All libraries imported")

✅ All libraries imported


## 2. Configuration

> ⚠️ **This is copied from your RAG agent notebook.** Make sure your `.env` file has:
> - `OPENAI_API_KEY`
> - `PINECONE_API_KEY`
> - `LANGSMITH_API_KEY` (same as `LANGCHAIN_API_KEY`)
> - `PINECONE_INDEX_NAME`
> - `PINECONE_NAMESPACE`

In [2]:
load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")

# LangSmith — traces will appear in this project
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "rag-agent-evaluation"   # 📌 You will see this in LangSmith
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY or ""

# Pinecone — same as your RAG agent
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "youtube-rag-mechanical-engineering")
NAMESPACE  = os.getenv("PINECONE_NAMESPACE",  "efficient-engineer-v3")

# Models — using mini to keep costs low
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL      = "gpt-4o-mini"   # agent model (cheap)
JUDGE_MODEL     = "gpt-4o-mini"   # LLM-as-judge model — swap to gpt-4o for higher accuracy

TOP_K = 3

print("✅ Configuration loaded")
print(f"   Chat Model   : {CHAT_MODEL}")
print(f"   Judge Model  : {JUDGE_MODEL}")
print(f"   Pinecone     : {INDEX_NAME} / {NAMESPACE}")
print(f"   LangSmith    : project = rag-agent-evaluation")

✅ Configuration loaded
   Chat Model   : gpt-4o-mini
   Judge Model  : gpt-4o-mini
   Pinecone     : youtube-rag-mechanical-engineering / efficient-engineer-v3
   LangSmith    : project = rag-agent-evaluation


## 3. Rebuild Your RAG Agent (copy-pasted from your notebook)

> 💡 **Why do we rebuild the agent here?**  
> The evaluation runner calls your agent function in a loop. It needs the agent to be defined in this notebook so it can wrap it with `@traceable` and send results to LangSmith.

In [3]:
# ── Initialize LLM + Embeddings + VectorStore ──────────────────────────────

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings, namespace=NAMESPACE)

print("✅ LLM, Embeddings, VectorStore initialised")

✅ LLM, Embeddings, VectorStore initialised


In [4]:
# ── Tool input schemas ──────────────────────────────────────────────────────

class SearchTranscriptsInput(BaseModel):
    query: str = Field(description="The search query to find relevant transcript content")

class GetVideoInfoInput(BaseModel):
    video_id: str = Field(description="The YouTube video ID to get information about")

class FindVideosInput(BaseModel):
    topic: str = Field(description="The topic to search for videos about")


# ── Tool functions ──────────────────────────────────────────────────────────

# We keep a module-level list to capture retrieved docs for evaluation
_last_retrieved_contexts: List[str] = []

def search_transcripts(query: str) -> str:
    """Search video transcripts for relevant content."""
    global _last_retrieved_contexts
    try:
        results = vectorstore.similarity_search(query, k=TOP_K)
        if not results:
            _last_retrieved_contexts = []
            return "No relevant information found."

        formatted_results = []
        _last_retrieved_contexts = []   # reset for this call

        for i, doc in enumerate(results, 1):
            metadata    = doc.metadata
            content     = doc.page_content
            video_id    = metadata.get('video_id',    metadata.get('videoId', 'unknown'))
            title       = metadata.get('video_title', metadata.get('title',   'Unknown Video'))
            start_time  = metadata.get('start_time',  metadata.get('start',   0))

            chunk_text = (
                f"[Result {i}]\n"
                f"Video: {title}\n"
                f"Time: {start_time}s\n"
                f"Content: {content}\n"
                f"Link: https://youtube.com/watch?v={video_id}&t={int(start_time)}s\n"
            )
            formatted_results.append(chunk_text)
            _last_retrieved_contexts.append(content)   # plain text for evaluators

        return "\n".join(formatted_results)
    except Exception as e:
        return f"Error searching transcripts: {str(e)}"


def get_video_info(video_id: str) -> str:
    """Get metadata about a specific video."""
    try:
        results = vectorstore.similarity_search(query="", k=1, filter={"video_id": video_id})
        if not results:
            return f"Video {video_id} not found in database."
        metadata = results[0].metadata
        return (
            f"Title: {metadata.get('video_title', metadata.get('title', 'Unknown'))}\n"
            f"Channel: {metadata.get('channel', 'Unknown')}\n"
            f"Video ID: {video_id}\n"
            f"Link: https://youtube.com/watch?v={video_id}"
        )
    except Exception as e:
        return f"Error getting video info: {str(e)}"


def find_videos(topic: str) -> str:
    """Find videos related to a specific topic."""
    try:
        results = vectorstore.similarity_search(topic, k=TOP_K * 2)
        if not results:
            return "No videos found on this topic."
        seen, videos = set(), []
        for doc in results:
            vid = doc.metadata.get('video_id', doc.metadata.get('videoId'))
            if vid and vid not in seen:
                seen.add(vid)
                videos.append({
                    'title':    doc.metadata.get('video_title', doc.metadata.get('title', 'Unknown')),
                    'channel':  doc.metadata.get('channel', 'Unknown'),
                    'video_id': vid
                })
            if len(videos) >= TOP_K:
                break
        formatted = []
        for i, v in enumerate(videos, 1):
            formatted.append(
                f"{i}. {v['title']}\n"
                f"   Channel: {v['channel']}\n"
                f"   Link: https://youtube.com/watch?v={v['video_id']}"
            )
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error finding videos: {str(e)}"


# ── Register tools ──────────────────────────────────────────────────────────

tools = [
    StructuredTool.from_function(
        func=search_transcripts,
        name="search_transcripts",
        description=(
            "Search YouTube video transcripts for information. Use this when the user asks "
            "questions about engineering concepts, definitions, or explanations."
        ),
        args_schema=SearchTranscriptsInput,
    ),
    StructuredTool.from_function(
        func=find_videos,
        name="find_videos",
        description=(
            "Find videos about a specific topic. Use this when the user asks 'which videos' "
            "or 'show me videos'."
        ),
        args_schema=FindVideosInput,
    ),
    StructuredTool.from_function(
        func=get_video_info,
        name="get_video_info",
        description="Get detailed metadata about a specific video by its ID.",
        args_schema=GetVideoInfoInput,
    ),
]

print(f"✅ {len(tools)} tools registered")

✅ 3 tools registered


In [5]:
# ── Agent prompt (same as your RAG notebook) ────────────────────────────────

system_message = """You are an expert mechanical engineering assistant with access to YouTube video transcripts.

Your role:
- Answer engineering questions using the transcript search tool
- Provide accurate, detailed technical explanations
- Always cite sources with timestamps when available
- Suggest relevant videos when appropriate

Guidelines:
- Use search_transcripts for concept explanations and definitions
- Use find_videos when users ask "which videos" or "show me videos"
- Always include YouTube links with timestamps in your responses
- Be concise but thorough in explanations
- If information isn't found, say so clearly

Current date: {current_date}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Build one shared agent executor (reused across all eval questions — saves cost)
agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,            # set True if you want to see tool calls
    handle_parsing_errors=True,
    max_iterations=3,
)

print("✅ Agent executor ready")

✅ Agent executor ready


## 4. Prediction Function (wraps your agent for LangSmith)

> 💡 **What is `@traceable`?**  
> It tells LangSmith to record every call to this function as a traced run. You will see these runs in your LangSmith project dashboard, and the evaluation scores will be attached to them.

In [6]:
@traceable(name="rag_agent_predict")
def predict_rag_answer(example: dict) -> dict:
    """
    Wrapper that LangSmith's `evaluate()` will call for each dataset example.
    Returns both 'answer' and 'contexts' so all 4 evaluators can use it.
    """
    global _last_retrieved_contexts
    _last_retrieved_contexts = []   # reset before each call

    question = example["question"]

    try:
        response = agent_executor.invoke({
            "input": question,
            "chat_history": [],
            "current_date": datetime.now().strftime("%Y-%m-%d"),
        })
        answer = response.get("output", "")
    except Exception as e:
        answer = f"Error: {str(e)}"

    return {
        "answer": answer,
        "contexts": _last_retrieved_contexts,   # docs retrieved by search_transcripts tool
    }


# Quick smoke test — makes sure the pipeline works before running full eval
test = predict_rag_answer({"question": "What is stress in engineering?"})
print("✅ Smoke test passed")
print(f"   Answer preview  : {test['answer'][:200]}...")
print(f"   Contexts found  : {len(test['contexts'])} chunks")

✅ Smoke test passed
   Answer preview  : In engineering, **stress** is defined as the internal force per unit area within materials that arises from externally applied forces, uneven heating, or permanent deformation, and it allows for the a...
   Contexts found  : 3 chunks


## 5. 📝 Create Evaluation Dataset in LangSmith

### ✏️ THIS IS WHAT YOU NEED TO EDIT

Add **5–10 question/answer pairs** that cover the topics in your video transcripts.

**Tips for writing good reference answers:**
- They don't need to be perfect word-for-word — the LLM judge scores semantic similarity
- Include key facts, definitions, or formulas the answer should contain
- Cover different question types: concept explanations, comparisons, "which videos" queries

> ⚠️ The dataset is created **once** in LangSmith. If you re-run this cell, it will try to create it again and fail. If that happens, just skip this cell — the dataset already exists.

In [7]:
# ────────────────────────────────────────────────────────────────────────────
# ✏️  EDIT THIS: Add your own Q&A pairs
# ────────────────────────────────────────────────────────────────────────────

DATASET_NAME = "mechanical-engineering-rag-eval"  # name shown in LangSmith

inputs = [
    "What is stress in engineering?",
    "What is the difference between engineering stress and true stress?",
    "Explain Young's modulus and why it matters in design.",
    "What is fatigue failure and how does it occur?",
    "What is the difference between stress and strain?",
    "Which videos discuss fatigue failure?",
    "What is yield strength and why is it important?",
    "What is the difference between ductile and brittle materials?",
    "What is Hooke's law and when does it apply?",
    "What is the factor of safety in engineering design?",
    # ✏️  Add more questions here...
]

outputs = [
    # ✏️  Write the expected 'correct' answer for each question above
    "Stress is the internal force per unit area within a material, calculated as force divided by cross-sectional area (σ = F/A). It is measured in Pascals (Pa) or psi.",
    "Engineering stress uses the original cross-sectional area, while true stress uses the instantaneous (changing) cross-sectional area during deformation. True stress is always higher than engineering stress after necking begins.",
    "Young's modulus (E) is the ratio of stress to strain in the elastic region (E = σ/ε). A higher modulus means a stiffer material that deforms less under load. It is critical in material selection to minimize elastic deformations.",
    "Fatigue failure occurs when a material is subjected to repeated cyclic loading below its ultimate strength. It is a three-stage process: crack initiation (usually at surface stress concentrations), crack propagation, and final fracture.",
    "Stress is the internal force per unit area (σ = F/A, units: Pa). Strain is the fractional change in length (ε = ΔL/L₀, dimensionless). Stress causes strain; their ratio in the elastic region gives Young's modulus.",
    "The video 'Understanding Fatigue Failure and S-N Curves' by The Efficient Engineer covers fatigue failure mechanisms and S-N curves.",
    "Yield strength is the stress at which a material begins to deform plastically. Below the yield strength, deformation is elastic and reversible, but above it the material will not return to its original shape. It is important in design because components must operate below the yield strength to avoid permanent deformation.",
    "Ductile materials can undergo significant plastic deformation before fracture, while brittle materials fracture with little or no plastic deformation. Ductile failure is usually gradual and visible, whereas brittle failure occurs suddenly without warning.",
    "Hooke's law states that stress is proportional to strain in the elastic region of a material (σ = Eε). It applies only while the material remains within its linear elastic limit, before yielding begins.",
    "The factor of safety (FoS) is the ratio of a material’s strength to the applied stress (FoS = strength / working stress). It is used in engineering design to provide a margin against uncertainty, material defects, and unexpected loads.",
    # ✏️  Add matching reference answers here...
]

assert len(inputs) == len(outputs), "Each question must have a matching reference answer!"

# ────────────────────────────────────────────────────────────────────────────

client = Client()

# Check if dataset already exists so we don't duplicate it
existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    print(f"ℹ️  Dataset '{DATASET_NAME}' already exists in LangSmith — skipping creation.")
    print(f"   Delete it in the LangSmith UI if you want to recreate it.")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Evaluation QA pairs for the Mechanical Engineering RAG Agent.",
    )
    client.create_examples(
        inputs=[{"question": q} for q in inputs],
        outputs=[{"answer": a} for a in outputs],
        dataset_id=dataset.id,
    )
    print(f"✅ Dataset '{DATASET_NAME}' created in LangSmith with {len(inputs)} examples.")
    print(f"   View it at: https://smith.langchain.com")

✅ Dataset 'mechanical-engineering-rag-eval' created in LangSmith with 10 examples.
   View it at: https://smith.langchain.com


## 6. Pull LLM-as-Judge Prompts from LangSmith Hub

These are standardised grader prompts maintained by LangChain. They tell the judge LLM exactly what to score and how.

| Prompt | Scores |
|--------|--------|
| `rag-answer-vs-reference` | Is the answer correct? (0 or 1) |
| `rag-answer-hallucination` | Is the answer grounded in retrieved docs? (0 or 1) |
| `rag-document-relevance` | Are the retrieved docs relevant? (0 or 1) |

In [8]:
grade_prompt_answer_accuracy  = hub.pull("langchain-ai/rag-answer-vs-reference")
grade_prompt_hallucinations   = hub.pull("langchain-ai/rag-answer-hallucination")
grade_prompt_doc_relevance    = hub.pull("langchain-ai/rag-document-relevance")

# One shared judge LLM instance
judge_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)

print("✅ LLM-as-judge prompts loaded from LangSmith Hub")

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Lib\site-packages\langsmith\client.py:5317: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


✅ LLM-as-judge prompts loaded from LangSmith Hub


## 7. Define the 4 Evaluator Functions

> 💡 Each evaluator receives:
> - `run` → the output produced by your RAG agent (`run.outputs`)
> - `example` → the ground-truth row from your dataset (`example.inputs`, `example.outputs`)
>
> Each evaluator returns a `{"key": ..., "score": 0_or_1}` dict. LangSmith plots these scores.

In [9]:
# ── Evaluator 1: Answer Accuracy (vs. reference answer) ────────────────────

def answer_accuracy_evaluator(run, example) -> dict:
    """
    Metric: answer_accuracy
    Compares the agent's answer against the reference answer in the dataset.
    Score 1 = correct/similar, Score 0 = wrong/missing key info.
    Requires: reference answer in your dataset.
    """
    input_question = example.inputs["question"]
    reference      = example.outputs["answer"]
    prediction     = run.outputs["answer"]

    grader = grade_prompt_answer_accuracy | judge_llm
    result = grader.invoke({
        "question":       input_question,
        "correct_answer": reference,
        "student_answer": prediction,
    })
    score = result["Score"]
    print(f"  [answer_accuracy]    Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_accuracy", "score": score}


# ── Evaluator 2: Answer Hallucination ──────────────────────────────────────

def answer_hallucination_evaluator(run, example) -> dict:
    """
    Metric: answer_hallucination
    Checks whether the answer is grounded in the retrieved documents.
    Score 1 = grounded (not hallucinated), Score 0 = hallucinated.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    contexts       = run.outputs.get("contexts", [])
    prediction     = run.outputs["answer"]

    if not contexts:
        print(f"  [hallucination]      Q: {input_question[:60]:<60} → N/A (no contexts)")
        return {"key": "answer_hallucination", "score": None}  # can't evaluate without docs

    context_str = "\n\n".join(contexts)
    grader = grade_prompt_hallucinations | judge_llm
    result = grader.invoke({
        "documents":      context_str,
        "student_answer": prediction,
    })
    score = result["Score"]
    print(f"  [answer_hallucination] Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_hallucination", "score": score}


# ── Evaluator 3: Document Relevance ────────────────────────────────────────

def document_relevance_evaluator(run, example) -> dict:
    """
    Metric: document_relevance
    Checks whether the retrieved documents are relevant to the question.
    Score 1 = relevant, Score 0 = irrelevant/off-topic.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    contexts       = run.outputs.get("contexts", [])

    if not contexts:
        print(f"  [document_relevance] Q: {input_question[:60]:<60} → N/A (no contexts)")
        return {"key": "document_relevance", "score": None}

    context_str = "\n\n".join(contexts)
    grader = grade_prompt_doc_relevance | judge_llm
    result = grader.invoke({
        "question":  input_question,
        "documents": context_str,
    })
    score = result["Score"]
    print(f"  [document_relevance] Q: {input_question[:60]:<60} → {score}")
    return {"key": "document_relevance", "score": score}


# ── Evaluator 4: Answer Helpfulness (custom prompt — no hub needed) ─────────

HELPFULNESS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict but fair evaluator assessing whether an AI assistant's answer "
     "adequately addresses the user's question.\n"
     "Score 1 if the answer directly and helpfully addresses the question.\n"
     "Score 0 if the answer is off-topic, vague, or fails to address the question.\n"
     "Respond ONLY with a JSON object like: {{\"Score\": 1}} or {{\"Score\": 0}}"),
    ("human",
     "Question: {question}\n\nAnswer: {answer}\n\nScore:")
])

from langchain_core.output_parsers import JsonOutputParser

def answer_helpfulness_evaluator(run, example) -> dict:
    """
    Metric: answer_helpfulness
    Checks whether the answer actually addresses what the user asked.
    Score 1 = helpful, Score 0 = unhelpful/off-topic.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    prediction     = run.outputs["answer"]

    grader = HELPFULNESS_PROMPT | judge_llm | JsonOutputParser()
    try:
        result = grader.invoke({"question": input_question, "answer": prediction})
        score  = result.get("Score", 0)
    except Exception:
        score = 0

    print(f"  [answer_helpfulness] Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_helpfulness", "score": score}


print("✅ 4 evaluator functions defined")

✅ 4 evaluator functions defined


## 8. 🚀 Run the Full Evaluation

This runs all 4 evaluators across every example in your dataset and logs everything to LangSmith.

> 📌 After running, go to **https://smith.langchain.com** → **Projects** → **rag-agent-evaluation** to see your results.

In [10]:
print("🚀 Starting evaluation — this will take a few minutes...")
print(f"   Dataset    : {DATASET_NAME}")
print(f"   Judge LLM  : {JUDGE_MODEL}")
print(f"   LangSmith  : project = rag-agent-evaluation")
print()

experiment_results = evaluate(
    predict_rag_answer,
    data=DATASET_NAME,
    evaluators=[
        answer_accuracy_evaluator,          # Metric 1: correct vs. reference
        answer_hallucination_evaluator,     # Metric 2: grounded in retrieved docs
        document_relevance_evaluator,       # Metric 3: docs relevant to question
        answer_helpfulness_evaluator,       # Metric 4: answer addresses question
    ],
    experiment_prefix="rag-agent-eval",
    metadata={
        "agent_model":  CHAT_MODEL,
        "judge_model":  JUDGE_MODEL,
        "pinecone_index": INDEX_NAME,
        "top_k":        TOP_K,
        "description":  "Full evaluation of Mechanical Engineering RAG Agent with 4 LLM-as-judge metrics",
    },
)

print("\n✅ Evaluation complete!")
print("📊 View your results at: https://smith.langchain.com")

🚀 Starting evaluation — this will take a few minutes...
   Dataset    : mechanical-engineering-rag-eval
   Judge LLM  : gpt-4o-mini
   LangSmith  : project = rag-agent-evaluation

View the evaluation results for experiment: 'rag-agent-eval-65d7ac03' at:
https://smith.langchain.com/o/4838b4b3-1897-4213-9b1c-62965e576710/datasets/32ddfeaf-1e77-4867-a459-a3549524c168/compare?selectedSessions=c24d6693-5be8-4f29-95ee-e976b3c547d9




0it [00:00, ?it/s]

  [answer_accuracy]    Q: What is the difference between engineering stress and true s → 1
  [answer_accuracy]    Q: What is fatigue failure and how does it occur?               → 1
  [answer_accuracy]    Q: Which videos discuss fatigue failure?                        → 1
  [answer_accuracy]    Q: Explain Young's modulus and why it matters in design.        → 1
  [answer_accuracy]    Q: What is the difference between ductile and brittle materials → 1
  [answer_accuracy]    Q: What is the factor of safety in engineering design?          → 0
  [answer_accuracy]    Q: What is Hooke's law and when does it apply?                  → 0
  [answer_accuracy]    Q: What is stress in engineering?                               → 1
  [answer_accuracy]    Q: What is yield strength and why is it important?              → 1
  [answer_accuracy]    Q: What is the difference between stress and strain?            → 1
  [answer_hallucination] Q: What is the difference between engineering stress and true s →

## 9. 📊 Print Local Summary of Results

In [ ]:
# Collect scores per metric from the experiment results
metric_scores: Dict[str, List[float]] = {}

for result in experiment_results:
    # experiment_results is iterable; each item has .feedback_results
    for feedback in result.get("evaluation_results", {}).get("results", []):
        key   = feedback.key
        score = feedback.score
        if score is not None:
            metric_scores.setdefault(key, []).append(score)

print("=" * 55)
print("📊 EVALUATION SUMMARY")
print("=" * 55)

if metric_scores:
    for metric, scores in sorted(metric_scores.items()):
        avg   = sum(scores) / len(scores)
        count = len(scores)
        bar   = "█" * int(avg * 20) + "░" * (20 - int(avg * 20))
        print(f"  {metric:<28} {bar}  {avg:.0%}  ({count} examples)")
else:
    print("  (Run the cell above first to populate results)")

print("=" * 55)
print("\n📌 Full results with traces: https://smith.langchain.com")

## 10. 🔁 (Optional) Run Individual Metrics Separately

If you want to run only one metric at a time (e.g., to debug), use the cells below.

In [ ]:
# ── Run ONLY hallucination check ───────────────────────────────────────────
# Uncomment and run if you want to isolate this metric

# experiment_hallucination = evaluate(
#     predict_rag_answer,
#     data=DATASET_NAME,
#     evaluators=[answer_hallucination_evaluator],
#     experiment_prefix="rag-agent-hallucination-only",
# )

In [ ]:
# ── Run ONLY document relevance ────────────────────────────────────────────

# experiment_doc_relevance = evaluate(
#     predict_rag_answer,
#     data=DATASET_NAME,
#     evaluators=[document_relevance_evaluator],
#     experiment_prefix="rag-agent-doc-relevance-only",
# )

---

## 📚 Guide: How to Read Your LangSmith Dashboard

After running Section 8, go to **https://smith.langchain.com**:

1. **Projects** tab → click **rag-agent-evaluation**
2. Click the **Experiments** tab to see the run named `rag-agent-eval-...`
3. Click into the experiment to see a table with one row per question and columns for each metric score
4. Click any row to see the **full trace** — which tool was called, what docs were retrieved, and how the answer was generated

### Interpreting scores (all metrics are 0 or 1)

| Score | Meaning |
|-------|---------|
| `answer_accuracy = 1` | Agent's answer matches the reference answer ✅ |
| `answer_accuracy = 0` | Answer is wrong or missing key information ❌ |
| `answer_hallucination = 1` | Answer is grounded in retrieved docs ✅ |
| `answer_hallucination = 0` | Answer contains info NOT in the retrieved docs ❌ |
| `document_relevance = 1` | Retrieved chunks are relevant to the question ✅ |
| `document_relevance = 0` | Retrieval returned off-topic chunks ❌ |
| `answer_helpfulness = 1` | Answer addresses what was asked ✅ |
| `answer_helpfulness = 0` | Answer is vague, off-topic, or unhelpful ❌ |

### What to do when scores are low

| Low metric | Likely cause | How to fix |
|------------|-------------|------------|
| `document_relevance` | Poor vector search | Increase `TOP_K`, improve chunking, add metadata filters |
| `answer_hallucination` | LLM ignores retrieved docs | Strengthen system prompt: "only use information from the retrieved documents" |
| `answer_accuracy` | Wrong facts | Check if the answer is in your Pinecone index; review chunking |
| `answer_helpfulness` | Off-topic answers | Improve agent routing logic; add more specific tool descriptions |

---

## 🔄 How to Re-evaluate After Making Changes

1. Make your change (e.g., update TOP_K, change the system prompt, improve chunking)
2. Re-run cells 3, 6, and 8 — LangSmith will create a **new experiment** automatically
3. In LangSmith, use **Compare Experiments** to see if scores improved